# Classification des clients
Construire un modèle permettant de prédire :_ y = churn_

In [25]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

In [2]:
data = pd.read_csv("../data/preprocessed_data.csv")
data.head()

,gender_Female,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen,Partner,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,Contract,Churn,tenure,MonthlyCharges,TotalCharges
0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0,1,...,0,0,0,0,1,0,0,0.013889,0.115423,0.003437
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,...,1,0,0,0,0,1,0,0.472222,0.385075,0.217564
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,...,0,0,0,0,1,0,1,0.027778,0.354229,0.012453
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0,...,1,1,0,0,0,1,0,0.625000,0.239303,0.211951
4,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0,0,...,0,0,0,0,1,0,1,0.027778,0.521891,0.017462


In [3]:
X = data.drop("Churn", axis=1)
y = data.Churn

display(X.shape)
display(y.shape)

(7043, 24)

(7043,)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=2,
    stratify=y
)

## Tester plusieurs modèles :

In [6]:
# Logistic Regression

lr_model = LogisticRegression(max_iter=100, random_state=1)

params = [
    {
        "penalty": ["l1", "l2"],
        "C": [0.001, 0.01, 0.1, 1, 10, 100],
        "solver": ["liblinear"],
        "class_weight": [None, "balanced"]
    },
    {
        "penalty": ["l2"],
        "C": [0.001, 0.01, 0.1, 1, 10, 100],
        "solver": ["lbfgs"],
        "class_weight": [None, "balanced"]
    },
    {
        "penalty": ["elasticnet"],
        "C": [0.001, 0.01, 0.1, 1, 10, 100],
        "solver": ["saga"],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
        "class_weight": [None, "balanced"]
    }
]

scoring = {
    "accuracy": "accuracy",
    "recall": "recall",
    "precision": "precision",
    "balanced_accuracy": "balanced_accuracy",
}

grid = GridSearchCV(
    lr_model,
    params,
    cv=5,
    scoring=scoring,
    refit="recall"
)

In [ ]:
grid.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.p

GridSearchCV(cv=5, estimator=LogisticRegression(random_state=1),
             param_grid=[{'C': [0.001, 0.01, 0.1, 1, 10, 100],
                          'class_weight': [None, 'balanced'],
                          'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
                         {'C': [0.001, 0.01, 0.1, 1, 10, 100],
                          'class_weight': [None, 'balanced'], 'penalty': ['l2'],
                          'solver': ['lbfgs']},
                         {'C': [0.001, 0.01, 0.1, 1, 10, 100],
                          'class_weight': [None, 'balanced'],
                          'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
                          'penalty': ['elasticnet'], 'solver': ['saga']}],
             refit='recall',
             scoring={'accuracy': 'accuracy',
                      'balanced_accuracy': 'balanced_accuracy',
                      'precision': 'precision', 'recall': 'recall'})

In [ ]:
display(grid.best_params_)
display(grid.best_score_)

{'C': 0.001,
 'class_weight': 'balanced',
 'l1_ratio': 0.3,
 'penalty': 'elasticnet',
 'solver': 'saga'}

np.float64(0.8762541806020068)

In [ ]:
# Decision Tree
dt_model = DecisionTreeClassifier(random_state=3)

params = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    dt_model,
    params,
    cv=5,
    scoring=scoring,
    refit="recall"
)

In [ ]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=3),
             param_grid={'class_weight': [None, 'balanced'],
                         'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [3, 5, 7, 10, None],
                         'min_samples_leaf': [1, 2, 5, 10],
                         'min_samples_split': [2, 5, 10, 20]},
             refit='recall',
             scoring={'accuracy': 'accuracy',
                      'balanced_accuracy': 'balanced_accuracy',
                      'precision': 'precision', 'recall': 'recall'})

In [ ]:
display(grid.best_params_)
display(grid.best_score_)

{'class_weight': 'balanced',
 'criterion': 'entropy',
 'max_depth': 5,
 'min_samples_leaf': 5,
 'min_samples_split': 2}

np.float64(0.8006688963210703)

In [ ]:
# Random Forest
rf_model = RandomForestClassifier(random_state=4)

params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    rf_model,
    params,
    cv=5,
    scoring=scoring,
    refit="recall"
)

In [ ]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=4),
             param_grid={'class_weight': [None, 'balanced'],
                         'max_depth': [None, 5, 10, 15, 20],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 5, 10],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             refit='recall',
             scoring={'accuracy': 'accuracy',
                      'balanced_accuracy': 'balanced_accuracy',
                      'precision': 'precision', 'recall': 'recall'})

In [ ]:
display(grid.best_params_)
display(grid.best_score_)

{'class_weight': 'balanced',
 'max_depth': 5,
 'max_features': 'sqrt',
 'min_samples_leaf': 2,
 'min_samples_split': 2,
 'n_estimators': 200}

np.float64(0.7879598662207358)

In [ ]:
# SVM
svc_model = SVC(random_state=5)

params = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "kernel": ['linear', 'poly', 'rbf', 'sigmoid'],
    "gamma": ["scale", "auto"],
    "class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    svc_model,
    params,
    cv=5,
    scoring=scoring,
    refit="recall"
)

In [ ]:
grid.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.p

GridSearchCV(cv=5, estimator=SVC(random_state=5),
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'class_weight': [None, 'balanced'],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'poly', 'rbf', 'sigmoid']},
             refit='recall',
             scoring={'accuracy': 'accuracy',
                      'balanced_accuracy': 'balanced_accuracy',
                      'precision': 'precision', 'recall': 'recall'})

In [ ]:
display(grid.best_params_)
display(grid.best_score_)

{'C': 0.001, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'poly'}

np.float64(0.9505016722408026)

In [7]:
# Gradient Boosting ou XGBoost
xgb_model = XGBClassifier(random_state=6)

params = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.3, 0.5, 1],
}

search = RandomizedSearchCV(
    xgb_model,
    params,
    cv=5,
    scoring=scoring,
    refit="recall",
)

In [8]:
search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=True,
                                           eval_metric=None, feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_constraint...
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'gamma': [0, 0.1, 0.3, 0.5, 1],
                                        'learning_rate': [0.01, 0.03, 0.05,
                                                          0.1],
                                        'max_depth': [3, 4, 5, 6, 8],
                                        'min_child_weight': [1, 3, 5, 7],
                                        'n_estimators': [100, 200, 300, 500],
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   refit='recall',
                   scoring={'accuracy': 'accuracy',
                            'balanced_accuracy': 'balanced_accuracy',
                            'precision': 'precision', 'recall': 'recall'})

In [10]:
display(search.best_params_)
display(search.best_score_)

{'subsample': 0.9,
 'n_estimators': 100,
 'min_child_weight': 1,
 'max_depth': 5,
 'learning_rate': 0.1,
 'gamma': 0.3,
 'colsample_bytree': 0.9}

np.float64(0.5170568561872909)

**Best models**

In [12]:
lr_model = LogisticRegression(
    max_iter=100,
    random_state=1,
    C=0.001,
    class_weight='balanced',
    l1_ratio=0.3,
    penalty='elasticnet',
    solver='saga'
)

lr_model.fit(X_train, y_train)

LogisticRegression(C=0.001, class_weight='balanced', l1_ratio=0.3,
                   penalty='elasticnet', random_state=1, solver='saga')

In [40]:
lr_predicted_y = lr_model.predict(X_test)
lr_cm = confusion_matrix(y_test, lr_predicted_y)
"""
Avec sklearn, confusion_matrix(y_test, y_pred) utilise par défaut les lignes pour
les vraies classes et les colonnes pour les classes prédites.
"""
print(f"""
Matrice de confusion:

{" " * 19}{"*" * 39}
{" " * 19}* {"won't churn":^16} * {"will churn":^16} *
{"*" * 58}
* {"won't churn":^16} * {lr_cm[0][0]:^16} * {lr_cm[0][1]:^16} *
{"*" * 58}
* {"will churn":^16} * {lr_cm[1][0]:^16} * {lr_cm[1][1]:^16} *
{"*" * 58}
""")


Matrice de confusion:

                   ***************************************
                   *   won't churn    *    will churn    *
**********************************************************
*   won't churn    *       607        *       428        *
**********************************************************
*    will churn    *        29        *       345        *
**********************************************************



In [13]:
dt_model = DecisionTreeClassifier(
    random_state=3,
    class_weight='balanced',
    criterion='entropy',
    max_depth=5,
    min_samples_leaf=5,
    min_samples_split=2
)

dt_model.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=5, min_samples_leaf=5, random_state=3)

In [41]:
dt_predicted_y = dt_model.predict(X_test)
dt_cm = confusion_matrix(y_test, dt_predicted_y)
"""
Avec sklearn, confusion_matrix(y_test, y_pred) utilise par défaut les lignes pour
les vraies classes et les colonnes pour les classes prédites.
"""
print(f"""
Matrice de confusion:

{" " * 19}{"*" * 39}
{" " * 19}* {"won't churn":^16} * {"will churn":^16} *
{"*" * 58}
* {"won't churn":^16} * {dt_cm[0][0]:^16} * {dt_cm[0][1]:^16} *
{"*" * 58}
* {"will churn":^16} * {dt_cm[1][0]:^16} * {dt_cm[1][1]:^16} *
{"*" * 58}
""")


Matrice de confusion:

                   ***************************************
                   *   won't churn    *    will churn    *
**********************************************************
*   won't churn    *       731        *       304        *
**********************************************************
*    will churn    *        63        *       311        *
**********************************************************



In [14]:
rf_model = RandomForestClassifier(
    random_state=4,
    class_weight='balanced',
    max_depth=5,
    max_features='sqrt',
    min_samples_leaf=2,
    min_samples_split=2,
    n_estimators=200
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=2,
                       n_estimators=200, random_state=4)

In [42]:
rf_predicted_y = rf_model.predict(X_test)
rf_cm = confusion_matrix(y_test, rf_predicted_y)
"""
Avec sklearn, confusion_matrix(y_test, y_pred) utilise par défaut les lignes pour
les vraies classes et les colonnes pour les classes prédites.
"""
print(f"""
Matrice de confusion:

{" " * 19}{"*" * 39}
{" " * 19}* {"won't churn":^16} * {"will churn":^16} *
{"*" * 58}
* {"won't churn":^16} * {rf_cm[0][0]:^16} * {rf_cm[0][1]:^16} *
{"*" * 58}
* {"will churn":^16} * {rf_cm[1][0]:^16} * {rf_cm[1][1]:^16} *
{"*" * 58}
""")


Matrice de confusion:

                   ***************************************
                   *   won't churn    *    will churn    *
**********************************************************
*   won't churn    *       773        *       262        *
**********************************************************
*    will churn    *        68        *       306        *
**********************************************************



In [15]:
svc_model = SVC(
    random_state=5,
    C=0.001,
    class_weight='balanced',
    gamma='scale',
    kernel='poly'
)

svc_model.fit(X_train, y_train)

SVC(C=0.001, class_weight='balanced', kernel='poly', random_state=5)

In [43]:
svc_predicted_y = svc_model.predict(X_test)
svc_cm = confusion_matrix(y_test, svc_predicted_y)
"""
Avec sklearn, confusion_matrix(y_test, y_pred) utilise par défaut les lignes pour
les vraies classes et les colonnes pour les classes prédites.
"""
print(f"""
Matrice de confusion:

{" " * 19}{"*" * 39}
{" " * 19}* {"won't churn":^16} * {"will churn":^16} *
{"*" * 58}
* {"won't churn":^16} * {svc_cm[0][0]:^16} * {svc_cm[0][1]:^16} *
{"*" * 58}
* {"will churn":^16} * {xgb_cm[1][0]:^16} * {svc_cm[1][1]:^16} *
{"*" * 58}
""")


Matrice de confusion:

                   ***************************************
                   *   won't churn    *    will churn    *
**********************************************************
*   won't churn    *       405        *       630        *
**********************************************************
*    will churn    *       177        *       363        *
**********************************************************



In [16]:
xgb_model = XGBClassifier(
    random_state=6,
    subsample=0.9,
    n_estimators=100,
    min_child_weight=1,
    max_depth=5,
    learning_rate=0.1,
    gamma=0.3,
    colsample_bytree=0.9
)

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=0.3, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=1, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [44]:
xgb_predicted_y = xgb_model.predict(X_test)
xgb_cm = confusion_matrix(y_test, xgb_predicted_y)
"""
Avec sklearn, confusion_matrix(y_test, y_pred) utilise par défaut les lignes pour
les vraies classes et les colonnes pour les classes prédites.
"""
print(f"""
Matrice de confusion:

{" " * 19}{"*" * 39}
{" " * 19}* {"won't churn":^16} * {"will churn":^16} *
{"*" * 58}
* {"won't churn":^16} * {xgb_cm[0][0]:^16} * {xgb_cm[0][1]:^16} *
{"*" * 58}
* {"will churn":^16} * {xgb_cm[1][0]:^16} * {xgb_cm[1][1]:^16} *
{"*" * 58}
""")


Matrice de confusion:

                   ***************************************
                   *   won't churn    *    will churn    *
**********************************************************
*   won't churn    *       937        *        98        *
**********************************************************
*    will churn    *       177        *       197        *
**********************************************************



In [46]:
print(f"""
{"*" * 83}
* {"Model\\Metric":^14} * {"Accuracy":^10} * {"Precision":^10} * {"Recall":^10} * {"F1-score":^10} * {"ROC-AUC":^10} *
{"*" * 83}
* {"lr_model":^14} * {accuracy_score(y_test, lr_predicted_y):^10.4} * {precision_score(y_test, lr_predicted_y):^10.4} * {recall_score(y_test, lr_predicted_y):^10.4} * {f1_score(y_test, lr_predicted_y):^10.4} * {roc_auc_score(y_test, lr_predicted_y):^10.4} *
{"*" * 83}
* {"dt_model":^14} * {accuracy_score(y_test, dt_predicted_y):^10.4} * {precision_score(y_test, dt_predicted_y):^10.4} * {recall_score(y_test, dt_predicted_y):^10.4} * {f1_score(y_test, dt_predicted_y):^10.4} * {roc_auc_score(y_test, dt_predicted_y):^10.4} *
{"*" * 83}
* {"rf_model":^14} * {accuracy_score(y_test, rf_predicted_y):^10.4} * {precision_score(y_test, rf_predicted_y):^10.4} * {recall_score(y_test, rf_predicted_y):^10.4} * {f1_score(y_test, rf_predicted_y):^10.4} * {roc_auc_score(y_test, rf_predicted_y):^10.4} *
{"*" * 83}
* {"svc_model":^14} * {accuracy_score(y_test, svc_predicted_y):^10.4} * {precision_score(y_test, svc_predicted_y):^10.4} * {recall_score(y_test, svc_predicted_y):^10.4} * {f1_score(y_test, svc_predicted_y):^10.4} * {roc_auc_score(y_test, svc_predicted_y):^10.4} *
{"*" * 83}
* {"xgb_model":^14} * {accuracy_score(y_test, xgb_predicted_y):^10.4} * {precision_score(y_test, xgb_predicted_y):^10.4} * {recall_score(y_test, xgb_predicted_y):^10.4} * {f1_score(y_test, xgb_predicted_y):^10.4} * {roc_auc_score(y_test, xgb_predicted_y):^10.4} *
{"*" * 83}
""")


***********************************************************************************
*  Model\Metric  *  Accuracy  * Precision  *   Recall   *  F1-score  *  ROC-AUC   *
***********************************************************************************
*    lr_model    *   0.6757   *   0.4463   *   0.9225   *   0.6016   *   0.7545   *
***********************************************************************************
*    dt_model    *   0.7395   *   0.5057   *   0.8316   *   0.6289   *   0.7689   *
***********************************************************************************
*    rf_model    *   0.7658   *   0.5387   *   0.8182   *   0.6497   *   0.7825   *
***********************************************************************************
*   svc_model    *   0.5451   *   0.3656   *   0.9706   *   0.5311   *   0.6809   *
***********************************************************************************
*   xgb_model    *   0.8048   *   0.6678   *   0.5267   *   0.5889   *   0.

comme notre priorité est de détecter le maximum de départs on opetra pour le modèle de svc.